# Alpha draft model experiment

Load normalized Dota 2 drafts, build PyTorch tensors, create the Alpha model, and prepare a train/validation loop.

In [ ]:
import json
import sys
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from scripts.ml.dataset import DatasetMaker
from scripts.ml.models.alpha import AlphaDraftModel

In [ ]:
DATASET_SIZE = 1_000
BATCH_SIZE = 64
VAL_FRACTION = 0.2
SEED = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
with DatasetMaker(env_file=str(PROJECT_ROOT / ".env")) as dataset:
    matches = dataset.fetch_normalized_match_drafts(DATASET_SIZE)

len(matches), matches[0]

In [ ]:
radiant_ids = torch.tensor([match.radiant_hero_ids for match in matches], dtype=torch.long)
dire_ids = torch.tensor([match.dire_hero_ids for match in matches], dtype=torch.long)

# DatasetMaker currently encodes winner_side as radiant=0, dire=1.
# AlphaDraftModel logits are positive for Radiant win, so labels must be radiant=1.0, dire=0.0.
labels = torch.tensor([1.0 if match.winner_side == 0 else 0.0 for match in matches], dtype=torch.float32)

radiant_ids.shape, dire_ids.shape, labels.shape, labels.float().mean()

In [ ]:
dataset_tensor = TensorDataset(radiant_ids, dire_ids, labels)

val_size = max(1, int(len(dataset_tensor) * VAL_FRACTION))
train_size = len(dataset_tensor) - val_size

generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset = random_split(dataset_tensor, [train_size, val_size], generator=generator)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

len(train_dataset), len(val_dataset)

In [ ]:
heroes_path = PROJECT_ROOT / "dotaconstants" / "build" / "heroes.json"
heroes = json.loads(heroes_path.read_text(encoding="utf-8"))
num_heroes = max(int(hero["id"]) for hero in heroes.values()) + 1

model = AlphaDraftModel(num_heroes=num_heroes, embedding_dim=32).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

num_heroes, model

In [ ]:
batch_radiant_ids, batch_dire_ids, batch_labels = next(iter(train_loader))
batch_radiant_ids = batch_radiant_ids.to(device)
batch_dire_ids = batch_dire_ids.to(device)
batch_labels = batch_labels.to(device)

model.train()
logits = model(batch_radiant_ids, batch_dire_ids)
loss = criterion(logits, batch_labels)
probs = torch.sigmoid(logits)

logits.shape, loss.item(), probs[:5].detach().cpu()

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_examples = 0

    for radiant_batch, dire_batch, label_batch in loader:
        radiant_batch = radiant_batch.to(device)
        dire_batch = dire_batch.to(device)
        label_batch = label_batch.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(radiant_batch, dire_batch)
        loss = criterion(logits, label_batch)
        loss.backward()
        optimizer.step()

        batch_size = label_batch.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size

    return total_loss / total_examples


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for radiant_batch, dire_batch, label_batch in loader:
        radiant_batch = radiant_batch.to(device)
        dire_batch = dire_batch.to(device)
        label_batch = label_batch.to(device)

        logits = model(radiant_batch, dire_batch)
        loss = criterion(logits, label_batch)
        preds = (torch.sigmoid(logits) >= 0.5).float()

        batch_size = label_batch.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (preds == label_batch).sum().item()
        total_examples += batch_size

    return {
        "loss": total_loss / total_examples,
        "accuracy": total_correct / total_examples,
    }

In [ ]:
# Run this cell when you are ready to start training.
# EPOCHS = 10
#
# for epoch in range(1, EPOCHS + 1):
#     train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
#     val_metrics = evaluate(model, val_loader, criterion, device)
#     print(
#         f"epoch={epoch:02d} "
#         f"train_loss={train_loss:.4f} "
#         f"val_loss={val_metrics['loss']:.4f} "
#         f"val_accuracy={val_metrics['accuracy']:.3f}"
#     )